In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # or "1", "0,1", etc.

import torch
print(torch.cuda.device_count())  # should show 1 if restricted

1


In [2]:
import pandas as pd
import evaluate
from sentence_transformers import SentenceTransformer
from tqdm import tqdm
from pathlib import Path
import json

/home/halim/.virtualenvs/experiment/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
def get_generated_questions(file_path):
    try:
        df = pd.read_json(file_path, lines=True, orient='records')
        contents = [item.replace("\nPertanyaan:  ", "").split("\n\nJawaban:")[0] for item in df['content']]
    except:
        df = pd.read_json(file_path, lines=True, orient='records')
        contents = df['generated_text']
    return contents

bleu = evaluate.load("bleu")
rouge = evaluate.load('rouge')

def compute_score(generated_list_of_content, task, dataset='squad'):
    if dataset == 'squad':
        reference = pd.read_json("~/Documents/halim/few-shot-aqg/dev_with_examples.jsonl", lines=True, orient='records')
        reference = reference[reference['is_impossible'] == False].reset_index(drop=True)
        if task == 'qg':
            reference = reference['question']
        elif task == 'ae':
            reference = reference['answer']
        else:
            return "the task can only be qg or ae"
    else:
        with open('../data/tydiqa-preprocesed-eval.json', 'r') as f:
            data = json.load(f)
        reference = [item['question'] for item in data]
        print(reference)
    


    bleu_score = bleu.compute(predictions=generated_list_of_content, references=reference)
    precision_keys = ['bleu1', 'bleu2', 'bleu3', 'bleu4']
    for i, key in enumerate(precision_keys):
        bleu_score[key] = bleu_score['precisions'][i]    
    del bleu_score['precisions']
    rouge_score = rouge.compute(predictions=generated_list_of_content, references=reference)

    model = SentenceTransformer("all-MiniLM-L6-v2")
    
    bert_scores = [model.similarity(model.encode(each), model.encode(reference[i]))[0] 
                                    for i, each in tqdm(
                                        enumerate(generated_list_of_content), 
                                        total=len(generated_list_of_content)
                                    )]
    average_bert_scores = {"bert_score": sum(bert_scores)/len(bert_scores)}

    return {**bleu_score, 
            **{k: float(v) for k, v in rouge_score.items()}, 
            **{k: float(v) for k, v in average_bert_scores.items()}}


In [4]:
list_files = sorted(list(Path(".").glob("*.jsonl")))
list_files = [item for item in list_files if 'question-generation' in item.name and 'SQuAD-id' in item.name]
list_files

[PosixPath('question-generation-idt5-base-qaqg-ae-noprefix-noprepend-42-SQuAD-id-checkpoint-8475.jsonl'),
 PosixPath('question-generation-idt5-base-qaqg-ae-yesprefix-noprepend-42-SQuAD-id-checkpoint-8475.jsonl'),
 PosixPath('question-generation-idt5-base-qaqg-ae-yesprefix-yesprepend-42-SQuAD-id-checkpoint-8475.jsonl'),
 PosixPath('question-generation-idt5-base-qaqg-noprefix-noprepend-42-SQuAD-id-checkpoint-23475.jsonl'),
 PosixPath('question-generation-idt5-base-qaqg-qg-noprefix-noprepend-42-SQuAD-id-checkpoint-15000.jsonl'),
 PosixPath('question-generation-idt5-base-qaqg-qg-yesprefix-noprepend-42-SQuAD-id-checkpoint-15000.jsonl'),
 PosixPath('question-generation-idt5-base-qaqg-qg-yesprefix-yesprepend-42-SQuAD-id-checkpoint-15000.jsonl'),
 PosixPath('question-generation-idt5-base-qaqg-yesprefix-noprepend-42-SQuAD-id-checkpoint-23475.jsonl'),
 PosixPath('question-generation-v1-12-SQuAD-id.jsonl'),
 PosixPath('question-generation-v1-42-SQuAD-id.jsonl'),
 PosixPath('question-generation-v1

In [5]:
result_df = pd.DataFrame([compute_score(get_generated_questions(item), 'qg') for item in list_files])
result_df['model'] = [item.name.split("question-generation-")[1].replace(".jsonl","") for item in list_files]
# variants = [item.name.split("few_")[1].replace(".jsonl","") for item in list_files]
# variants_type = [
#     "randmix" if "rand_ex_mix" in x
#     else "randfix" if "rand_ex_same" in x
#     else "similar" if "sim_ex_question" in x
#     else None
#     for x in variants
# ]

# variants_shot = [
#     "3" if "3shots" in x
#     else "5" if "5shots" in x
#     else None
#     for x in variants
# ]

# result_df['variant_type'] = variants_type
# result_df['variant_shot'] = variants_shot

100%|██████████| 5928/5928 [00:18<00:00, 315.36it/s]


In [6]:
result_df

,bleu,brevity_penalty,length_ratio,translation_length,reference_length,bleu1,bleu2,bleu3,bleu4,rouge1,rouge2,rougeL,rougeLsum,bert_score,model
0,0.009273,1.0,7.413128,443268,59795,0.039037,0.012700,0.005470,0.002726,0.088447,0.028123,0.076500,0.076441,0.459429,idt5-base-qaqg-ae-noprefix-noprepend-42-SQuAD-...
1,0.009113,1.0,7.606271,454817,59795,0.039024,0.012631,0.005368,0.002606,0.087629,0.028107,0.076009,0.075952,0.459092,idt5-base-qaqg-ae-yesprefix-noprepend-42-SQuAD...
2,0.009113,1.0,7.606271,454817,59795,0.039024,0.012631,0.005368,0.002606,0.087629,0.028107,0.076009,0.075952,0.459092,idt5-base-qaqg-ae-yesprefix-yesprepend-42-SQuA...
3,0.010152,1.0,6.079689,363535,59795,0.046326,0.013870,0.005838,0.002832,0.111454,0.035335,0.097641,0.097592,0.483151,idt5-base-qaqg-noprefix-noprepend-42-SQuAD-id-...
4,0.056095,1.0,1.140564,68200,59795,0.258416,0.065824,0.031521,0.018466,0.229359,0.078028,0.208757,0.208785,0.593942,idt5-base-qaqg-qg-noprefix-noprepend-42-SQuAD-...
5,0.055210,1.0,1.162907,69536,59795,0.255479,0.065306,0.031259,0.017816,0.228868,0.077193,0.208636,0.208568,0.592840,idt5-base-qaqg-qg-yesprefix-noprepend-42-SQuAD...
6,0.166386,1.0,1.027561,61443,59795,0.445844,0.203098,0.116059,0.072929,0.445518,0.226297,0.409140,0.409287,0.717021,idt5-base-qaqg-qg-yesprefix-yesprepend-42-SQuA...
7,0.018563,1.0,3.820286,228434,59795,0.087631,0.023447,0.010449,0.005531,0.170819,0.057480,0.146377,0.146329,0.559849,idt5-base-qaqg-yesprefix-noprepend-42-SQuAD-id...
8,0.061397,1.0,2.538055,151763,59795,0.191931,0.077752,0.040784,0.023347,0.380914,0.189458,0.339269,0.339567,0.686163,v1-12-SQuAD-id
9,0.148141,1.0,1.155046,69066,59795,0.405366,0.181048,0.102727,0.063882,0.439574,0.224219,0.401069,0.401423,0.714617,v1-42-SQuAD-id


In [7]:
list_files_tydiqa = sorted(list(Path(".").glob("*.jsonl")))
list_files_tydiqa = [item for item in list_files_tydiqa if 'question-generation' in item.name and 'TydiQA-id' in item.name]
list_files_tydiqa

[PosixPath('question-generation-idt5-base-qaqg-ae-noprefix-noprepend-42-TydiQA-id-checkpoint-5705.jsonl'),
 PosixPath('question-generation-idt5-base-qaqg-ae-yesprefix-noprepend-42-TydiQA-id-checkpoint-5705.jsonl'),
 PosixPath('question-generation-idt5-base-qaqg-noprefix-noprepend-42-TydiQA-id-checkpoint-5705.jsonl'),
 PosixPath('question-generation-idt5-base-qaqg-qg-noprefix-noprepend-42-TydiQA-id-checkpoint-2855.jsonl'),
 PosixPath('question-generation-idt5-base-qaqg-qg-yesprefix-noprepend-42-TydiQA-id-checkpoint-2855.jsonl'),
 PosixPath('question-generation-idt5-base-qaqg-qg-yesprefix-yesprepend-42-TydiQA-id-checkpoint-2855.jsonl'),
 PosixPath('question-generation-idt5-base-qaqg-yesprefix-noprepend-42-TydiQA-id-checkpoint-5705.jsonl'),
 PosixPath('question-generation-v1-12-TydiQA-id.jsonl'),
 PosixPath('question-generation-v1-42-TydiQA-id.jsonl'),
 PosixPath('question-generation-v1-72-TydiQA-id.jsonl'),
 PosixPath('question-generation-v2-12-TydiQA-id.jsonl'),
 PosixPath('question-gen

In [8]:
result_df_tydiqa = pd.DataFrame([compute_score(get_generated_questions(item), 'qg', 'tydiqa') for item in list_files_tydiqa])
result_df_tydiqa['model'] = [item.name.split("question-generation-")[1].replace(".jsonl","") for item in list_files_tydiqa]

['Siapakah yang menemuka benua Amerika ?', 'Dimanakah letak Donggala ?', 'Siapa bapak Teknik industri?', 'Kapan Penghulu Rasyid meninggal ?', 'seberapa luas kah samudera pasifik?', 'apakah yang dimaksud denga geisha ?', 'Kapan Bank BCA mengeluarkan kartu debit?', 'Dimana kantor pusat General Motors?', 'Berapa luas kota Blitar?', 'Siapa yang menciptakan serial manga Crows?', 'siapakah karakter utama serial anime dan manga Eyeshield 21?', 'Siapakah yang merumuskan naskah proklamasi ?', 'Bagaimanakah sistem pemerintahan di Jepang ?', 'kapankah Gerakan Pemuda Ansor didirikan?', 'Apakah yang diceritakan dalam The Years of Rice and Salt?', 'apakah pendidikan terakhir  Budi Susilo Soepandji?', 'dimanakah letak Cekungan Tarim?', 'terbuat dari apakah Genta ?', 'Siapakah R.L. Stine?', 'Siapakah yang menggagas Determinisme biologis?', 'Apakah nama lagu kebangsaan Jepang?', 'Dimana Konsili Kartago diadakan?', 'berapakah luas Bendung Katulampa?', 'Apa nama ilmiah tumbuhan kaktus ?', 'Kapan perahu p

100%|██████████| 565/565 [00:01<00:00, 321.49it/s]


['Siapakah yang menemuka benua Amerika ?', 'Dimanakah letak Donggala ?', 'Siapa bapak Teknik industri?', 'Kapan Penghulu Rasyid meninggal ?', 'seberapa luas kah samudera pasifik?', 'apakah yang dimaksud denga geisha ?', 'Kapan Bank BCA mengeluarkan kartu debit?', 'Dimana kantor pusat General Motors?', 'Berapa luas kota Blitar?', 'Siapa yang menciptakan serial manga Crows?', 'siapakah karakter utama serial anime dan manga Eyeshield 21?', 'Siapakah yang merumuskan naskah proklamasi ?', 'Bagaimanakah sistem pemerintahan di Jepang ?', 'kapankah Gerakan Pemuda Ansor didirikan?', 'Apakah yang diceritakan dalam The Years of Rice and Salt?', 'apakah pendidikan terakhir  Budi Susilo Soepandji?', 'dimanakah letak Cekungan Tarim?', 'terbuat dari apakah Genta ?', 'Siapakah R.L. Stine?', 'Siapakah yang menggagas Determinisme biologis?', 'Apakah nama lagu kebangsaan Jepang?', 'Dimana Konsili Kartago diadakan?', 'berapakah luas Bendung Katulampa?', 'Apa nama ilmiah tumbuhan kaktus ?', 'Kapan perahu p

100%|██████████| 565/565 [00:01<00:00, 320.82it/s]


['Siapakah yang menemuka benua Amerika ?', 'Dimanakah letak Donggala ?', 'Siapa bapak Teknik industri?', 'Kapan Penghulu Rasyid meninggal ?', 'seberapa luas kah samudera pasifik?', 'apakah yang dimaksud denga geisha ?', 'Kapan Bank BCA mengeluarkan kartu debit?', 'Dimana kantor pusat General Motors?', 'Berapa luas kota Blitar?', 'Siapa yang menciptakan serial manga Crows?', 'siapakah karakter utama serial anime dan manga Eyeshield 21?', 'Siapakah yang merumuskan naskah proklamasi ?', 'Bagaimanakah sistem pemerintahan di Jepang ?', 'kapankah Gerakan Pemuda Ansor didirikan?', 'Apakah yang diceritakan dalam The Years of Rice and Salt?', 'apakah pendidikan terakhir  Budi Susilo Soepandji?', 'dimanakah letak Cekungan Tarim?', 'terbuat dari apakah Genta ?', 'Siapakah R.L. Stine?', 'Siapakah yang menggagas Determinisme biologis?', 'Apakah nama lagu kebangsaan Jepang?', 'Dimana Konsili Kartago diadakan?', 'berapakah luas Bendung Katulampa?', 'Apa nama ilmiah tumbuhan kaktus ?', 'Kapan perahu p

...

100%|██████████| 565/565 [00:01<00:00, 325.00it/s]


['Siapakah yang menemuka benua Amerika ?', 'Dimanakah letak Donggala ?', 'Siapa bapak Teknik industri?', 'Kapan Penghulu Rasyid meninggal ?', 'seberapa luas kah samudera pasifik?', 'apakah yang dimaksud denga geisha ?', 'Kapan Bank BCA mengeluarkan kartu debit?', 'Dimana kantor pusat General Motors?', 'Berapa luas kota Blitar?', 'Siapa yang menciptakan serial manga Crows?', 'siapakah karakter utama serial anime dan manga Eyeshield 21?', 'Siapakah yang merumuskan naskah proklamasi ?', 'Bagaimanakah sistem pemerintahan di Jepang ?', 'kapankah Gerakan Pemuda Ansor didirikan?', 'Apakah yang diceritakan dalam The Years of Rice and Salt?', 'apakah pendidikan terakhir  Budi Susilo Soepandji?', 'dimanakah letak Cekungan Tarim?', 'terbuat dari apakah Genta ?', 'Siapakah R.L. Stine?', 'Siapakah yang menggagas Determinisme biologis?', 'Apakah nama lagu kebangsaan Jepang?', 'Dimana Konsili Kartago diadakan?', 'berapakah luas Bendung Katulampa?', 'Apa nama ilmiah tumbuhan kaktus ?', 'Kapan perahu p

100%|██████████| 565/565 [00:01<00:00, 318.42it/s]


['Siapakah yang menemuka benua Amerika ?', 'Dimanakah letak Donggala ?', 'Siapa bapak Teknik industri?', 'Kapan Penghulu Rasyid meninggal ?', 'seberapa luas kah samudera pasifik?', 'apakah yang dimaksud denga geisha ?', 'Kapan Bank BCA mengeluarkan kartu debit?', 'Dimana kantor pusat General Motors?', 'Berapa luas kota Blitar?', 'Siapa yang menciptakan serial manga Crows?', 'siapakah karakter utama serial anime dan manga Eyeshield 21?', 'Siapakah yang merumuskan naskah proklamasi ?', 'Bagaimanakah sistem pemerintahan di Jepang ?', 'kapankah Gerakan Pemuda Ansor didirikan?', 'Apakah yang diceritakan dalam The Years of Rice and Salt?', 'apakah pendidikan terakhir  Budi Susilo Soepandji?', 'dimanakah letak Cekungan Tarim?', 'terbuat dari apakah Genta ?', 'Siapakah R.L. Stine?', 'Siapakah yang menggagas Determinisme biologis?', 'Apakah nama lagu kebangsaan Jepang?', 'Dimana Konsili Kartago diadakan?', 'berapakah luas Bendung Katulampa?', 'Apa nama ilmiah tumbuhan kaktus ?', 'Kapan perahu p

100%|██████████| 565/565 [00:01<00:00, 326.53it/s]


In [9]:
result_df_tydiqa

,bleu,brevity_penalty,length_ratio,translation_length,reference_length,bleu1,bleu2,bleu3,bleu4,rouge1,rouge2,rougeL,rougeLsum,bert_score,model
0,0.008023,1.000000,1.449090,5337,3683,0.046468,0.010897,0.003963,0.002065,0.061819,0.017201,0.060309,0.060283,0.351681,idt5-base-qaqg-ae-noprefix-noprepend-42-TydiQA...
1,0.003552,1.000000,1.427641,5258,3683,0.041841,0.007671,0.001896,0.000262,0.056569,0.013367,0.055176,0.054866,0.338775,idt5-base-qaqg-ae-yesprefix-noprepend-42-TydiQ...
2,0.075385,1.000000,1.308173,4818,3683,0.181196,0.086292,0.055957,0.036912,0.208614,0.118236,0.206429,0.206001,0.495986,idt5-base-qaqg-noprefix-noprepend-42-TydiQA-id...
3,0.200307,0.957017,0.957915,3528,3683,0.472222,0.241309,0.155129,0.108565,0.421266,0.245741,0.418435,0.418485,0.712640,idt5-base-qaqg-qg-noprefix-noprepend-42-TydiQA...
4,0.187002,0.913076,0.916644,3376,3683,0.489336,0.239772,0.149154,0.100535,0.417818,0.231831,0.413089,0.413219,0.716007,idt5-base-qaqg-qg-yesprefix-noprepend-42-TydiQ...
5,0.207185,0.989355,0.989411,3644,3683,0.484358,0.243910,0.155529,0.104669,0.448582,0.250748,0.440564,0.440567,0.739036,idt5-base-qaqg-qg-yesprefix-yesprepend-42-Tydi...
6,0.224833,0.941613,0.943253,3474,3683,0.510363,0.271915,0.179608,0.130410,0.455641,0.279247,0.451643,0.451329,0.741827,idt5-base-qaqg-yesprefix-noprepend-42-TydiQA-i...
7,0.182778,1.000000,1.038284,3824,3683,0.457374,0.222154,0.133630,0.082198,0.438945,0.246476,0.431507,0.431785,0.735816,v1-12-TydiQA-id
8,0.106722,1.000000,1.354331,4988,3683,0.309142,0.129098,0.072836,0.044627,0.342852,0.177484,0.335083,0.335450,0.671329,v1-42-TydiQA-id
9,0.194075,1.000000,1.019549,3755,3683,0.461784,0.231348,0.142476,0.093204,0.437138,0.248171,0.430286,0.430126,0.729055,v1-72-TydiQA-id
